# Assignment 6 — Image Smoothing Filters

**Aim:** Smooth an input image using:
1. Unweighted averaging filter
2. Weighted filter, using \(h(x,y)=\max(|x|,|y|)\)
3. Gaussian filter

The program asks the user for the filter size, padding method, and Gaussian sigma. It displays the original image and the three smoothed results.

> **Weighted-filter interpretation:** The assignment writes `max(|x|.|y|)`. This notebook interprets that as `max(|x|, |y|)` for the spatial weight at each kernel coordinate. The weights are normalized so their sum is 1. If your instructor intended a different interpretation, confirm the formula with them.


## 1. Import required libraries

We use:
- **OpenCV (`cv2`)** to read and process images.
- **NumPy** for creating kernels and numerical operations.
- **Matplotlib** to display images.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Libraries imported successfully.")

## 2. Load the input image

Enter the path of an image. For a Google Colab session, you can upload an image when prompted. In a local Jupyter/VS Code setup, enter the image filename or full path.

The image is converted from OpenCV's BGR format to RGB for correct display.


In [ ]:
def load_image():
    try:
        from google.colab import files
        print("Upload an image file:")
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No image was uploaded.")
        image_path = next(iter(uploaded))
    except ImportError:
        image_path = input("Enter image path (example: image.jpg): ").strip()

    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(
            f"Could not read '{image_path}'. Check the file path and image format."
        )
    return image

image_bgr = load_image()
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(7, 5))
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")
plt.show()

print("Image shape:", image_rgb.shape)

## 3. Get filter parameters from the user

The filter size must be a positive odd integer (3, 5, 7, ...), so that the kernel has a well-defined center pixel.

Padding options:
- `replicate`: extend the image by repeating its border pixels.
- `zero`: add zero-valued pixels around the image.
- `reflect`: mirror the image at its borders.


In [ ]:
def get_odd_kernel_size():
    while True:
        try:
            size = int(input("Enter filter size (odd integer, e.g. 3 or 5): "))
            if size > 0 and size % 2 == 1:
                return size
            print("Please enter a positive odd integer.")
        except ValueError:
            print("Please enter a valid integer.")

def get_padding_method():
    valid = {"replicate", "zero", "reflect"}
    while True:
        method = input(
            "Choose padding [replicate / zero / reflect]: "
        ).strip().lower()
        if method in valid:
            return method
        print("Invalid choice. Select replicate, zero, or reflect.")

kernel_size = get_odd_kernel_size()
padding_method = get_padding_method()

while True:
    try:
        sigma = float(input("Enter Gaussian sigma (positive number, e.g. 1.0): "))
        if sigma > 0:
            break
        print("Sigma must be greater than zero.")
    except ValueError:
        print("Please enter a valid number.")

print("\nParameters selected:")
print("Filter size:", kernel_size)
print("Padding:", padding_method)
print("Gaussian sigma:", sigma)

## 4. Padding and filtering helper

A filter is applied by moving the kernel over each pixel neighborhood and calculating the weighted sum. Padding handles neighborhoods at the image boundaries.

This helper supports grayscale and color images.


In [ ]:
def get_cv2_border_type(method):
    if method == "replicate":
        return cv2.BORDER_REPLICATE
    elif method == "zero":
        return cv2.BORDER_CONSTANT
    elif method == "reflect":
        return cv2.BORDER_REFLECT
    else:
        raise ValueError("Unsupported padding method.")

def apply_kernel(image, kernel, padding_method):
    border_type = get_cv2_border_type(padding_method)
    # filter2D applies the kernel independently to each color channel.
    result = cv2.filter2D(
        image,
        ddepth=-1,
        kernel=kernel.astype(np.float32),
        borderType=border_type
    )
    return result

## 5. Unweighted averaging filter

Every element of the averaging kernel has the same value:

\[
h(x,y)=\frac{1}{mn}
\]

For a square filter of size \(k\times k\), each element is \(1/k^2\). This replaces each pixel with the average of the pixels in its neighborhood.


In [ ]:
def create_averaging_kernel(size):
    kernel = np.ones((size, size), dtype=np.float32)
    kernel = kernel / (size * size)
    return kernel

averaging_kernel = create_averaging_kernel(kernel_size)
averaging_result = apply_kernel(image_rgb, averaging_kernel, padding_method)

print("Averaging kernel:")
print(averaging_kernel)

## 6. Weighted filter

For each kernel coordinate relative to its center, this implementation uses:

\[
w(x,y)=\max(|x|,|y|)
\]

The weights are normalized by their total sum. The center weight is set to 1 so the kernel remains valid and gives the center pixel a nonzero contribution. This produces a center-inclusive spatial weighting kernel.


In [ ]:
def create_weighted_kernel(size):
    center = size // 2
    kernel = np.zeros((size, size), dtype=np.float32)

    for row in range(size):
        for col in range(size):
            x = col - center
            y = row - center
            kernel[row, col] = max(abs(x), abs(y))

    # Give the center pixel a nonzero weight.
    kernel[center, center] = 1.0

    total = np.sum(kernel)
    if total == 0:
        kernel[center, center] = 1.0
        total = 1.0

    return kernel / total

weighted_kernel = create_weighted_kernel(kernel_size)
weighted_result = apply_kernel(image_rgb, weighted_kernel, padding_method)

print("Weighted kernel:")
print(weighted_kernel)
print("Sum of weights:", np.sum(weighted_kernel))

## 7. Gaussian filter

The Gaussian kernel assigns weights according to the Gaussian function:

\[
G(x,y)=\frac{1}{2\pi\sigma^2}
\exp\left(-\frac{x^2+y^2}{2\sigma^2}\right)
\]

Here, \(\sigma\) controls the spread of the blur. A larger sigma generally produces stronger smoothing. The kernel is normalized so that all its weights sum to 1.


In [ ]:
def create_gaussian_kernel(size, sigma):
    center = size // 2
    kernel = np.zeros((size, size), dtype=np.float32)

    for row in range(size):
        for col in range(size):
            x = col - center
            y = row - center
            exponent = -(x*x + y*y) / (2 * sigma * sigma)
            kernel[row, col] = np.exp(exponent)

    kernel = kernel / np.sum(kernel)
    return kernel

gaussian_kernel = create_gaussian_kernel(kernel_size, sigma)
gaussian_result = apply_kernel(image_rgb, gaussian_kernel, padding_method)

print("Gaussian kernel:")
print(gaussian_kernel)
print("Sum of weights:", np.sum(gaussian_kernel))

## 8. Display and compare all results

The following output displays the original image and the results of the three filters.


In [ ]:
plt.figure(figsize=(15, 10))

images = [
    (image_rgb, "Original Image"),
    (averaging_result, "Unweighted Averaging Filter"),
    (weighted_result, "Weighted Filter"),
    (gaussian_result, f"Gaussian Filter (sigma={sigma})")
]

for index, (img, title) in enumerate(images, start=1):
    plt.subplot(2, 2, index)
    plt.imshow(np.clip(img, 0, 255).astype(np.uint8))
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

## 9. Save the output images (optional)

The results are saved in the current working directory. You can change the filenames if required.


In [ ]:
cv2.imwrite(
    "averaging_output.png",
    cv2.cvtColor(averaging_result, cv2.COLOR_RGB2BGR)
)
cv2.imwrite(
    "weighted_output.png",
    cv2.cvtColor(weighted_result, cv2.COLOR_RGB2BGR)
)
cv2.imwrite(
    "gaussian_output.png",
    cv2.cvtColor(gaussian_result, cv2.COLOR_RGB2BGR)
)

print("All three output images have been saved successfully.")

## 10. Conclusion

In this assignment, an input image was smoothed using:
1. An unweighted averaging filter, where all neighborhood pixels have equal weights.
2. A weighted filter, where weights depend on the distance of kernel coordinates from the center according to the selected formula.
3. A Gaussian filter, where weights follow a Gaussian distribution controlled by sigma.

The filter size, padding method, and Gaussian sigma are supplied by the user.